In [ ]:
import warnings

warnings.filterwarnings(
    "ignore",
    message="pkg_resources is deprecated as an API",
    category=UserWarning,
)

In [ ]:
import pymc as pm
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import arviz as az
import geopandas as gpd
from keplergl import KeplerGl
from helpers import ( prep_the_data, 
                      export_puma_kepler, 
                      make_daily_table_for_model_with_nta,
                      load_idata,

                      export_idata,
                      compare_models_loo_waic,
                      loo_diagnostics,
                        kepler_typical_week_from_dow_complaint
                    )




In [ ]:
df_puma = pd.read_parquet(
    "../data/processed/features/puma_noise_counts.parquet"
)


In [ ]:
df_puma = prep_the_data(df_puma)

In [ ]:
df_puma_2022__2024 = df_puma.loc[
    (df_puma["created_bucket"] >= "2022-01-01") &
    (df_puma["created_bucket"] <  "2024-12-31") &
     (df_puma["time_of_day"] ==  "night")
].copy()


In [ ]:
df_puma_2022__2024.head()

In [ ]:
daily_df, coords = make_daily_table_for_model_with_nta(
    df_puma_2022__2024,
    complaint_col="descriptor_group",
)

In [ ]:
# daily_df has: puma, nta_name, puma_idx, nta_idx, dow_idx, daily_count

# Build a unique mapping: puma_idx -> nta_idx
puma_nta_map = (
    daily_df[["puma_idx", "nta_idx"]]
    .drop_duplicates()
    .sort_values("puma_idx")
)

# sanity check: each puma maps to exactly 1 nta
assert puma_nta_map["puma_idx"].is_unique, "A PUMA maps to multiple NTAs (check your join)."

puma_to_nta_idx = puma_nta_map["nta_idx"].to_numpy()  # shape (n_puma,)


In [ ]:
# -----------------------------
# Data for the model
# -----------------------------
y = daily_df["daily_count"].to_numpy(dtype=int)
puma_idx_obs = daily_df["puma_idx"].to_numpy(dtype=int)
dow_idx_obs  = daily_df["dow_idx"].to_numpy(dtype=int)

In [ ]:

with pm.Model(coords=coords) as m3_nta:

    # NTA-level weekday baseline on log scale
    mu_nta = pm.Normal("mu_nta", mu=0.0, sigma=1.5, dims=("nta", "dow"))

    # PUMA-level deviation around its NTA baseline
    sigma_puma = pm.Exponential("sigma_puma", 1.0)
    delta_puma = pm.Normal("delta_puma", mu=0.0, sigma=sigma_puma, dims=("puma", "dow"))

    # Build log-rate for every (puma, dow) using PUMA->NTA mapping (NOT per-observation indexing)
    log_lambda = pm.Deterministic(
        "log_lambda",
        mu_nta[puma_to_nta_idx, :] + delta_puma,
        dims=("puma", "dow"),
    )

    lam = pm.Deterministic("lam", pm.math.exp(log_lambda), dims=("puma", "dow"))

    # Likelihood: each observation picks its (puma, dow)
    y_obs = pm.Poisson("y_obs", mu=lam[puma_idx_obs, dow_idx_obs], observed=y)

    idata_nta = pm.sample(
        draws=1000,
        tune=1000,
        chains=4,
        target_accept=0.9,
        random_seed=42,
        idata_kwargs={"log_likelihood": True},
    )

In [ ]:
# compare both modelss

In [ ]:
idata2 = load_idata("../data/processed/models/model2_pois_idata.nc")

In [ ]:
loo_table, waic_table = compare_models_loo_waic(idata2, idata_nta, m2_name="Poisson PUMA×DOW", m3_name="Poisson + NTA pooling")


In [ ]:
loo2 = loo_diagnostics(idata2, name="Model 2")
loo3 = loo_diagnostics(idata3, name="Model 3")


In [ ]:
# export for visualization

In [ ]:
# --- Posterior draws of lambda[puma, dow]
lam_post = idata_nta.posterior["lam"]  # dims: chain, draw, puma, dow

# Posterior mean
lam_mean = (
    lam_post
    .mean(dim=("chain", "draw"))
    .to_dataframe(name="lam_mean")
    .reset_index()
)

# 90% HDI
hdi_ds = az.hdi(lam_post, hdi_prob=0.9)
hdi_df = (
    hdi_ds["lam"]
    .to_dataframe(name="lam_hdi")
    .reset_index()
    .pivot_table(
        index=["puma", "dow"],
        columns="hdi",
        values="lam_hdi"
    )
    .reset_index()
    .rename(columns={"lower": "lam_low_90", "higher": "lam_high_90"})
)

# Combine
df_post = lam_mean.merge(hdi_df, on=["puma", "dow"], how="left")
df_post["lam_width_90"] = df_post["lam_high_90"] - df_post["lam_low_90"]

df_post.head()


In [ ]:
# --- Map weekday → synthetic date
dow_to_date = {
    "Monday":    "2000-01-03",
    "Tuesday":   "2000-01-04",
    "Wednesday": "2000-01-05",
    "Thursday":  "2000-01-06",
    "Friday":    "2000-01-07",
    "Saturday":  "2000-01-08",
    "Sunday":    "2000-01-09",
}

df_post["date"] = pd.to_datetime(
    df_post["dow"].map(dow_to_date),
    errors="coerce"
).astype("datetime64[ns]")


In [ ]:
gdf_kepler = kepler_typical_week_from_dow_complaint(
    df_post,
    puma_geojson_path="../data/raw/nyc/geographies/nyc_pumas_2020.geojson",
    out_path="data/processed/kepler/puma_typical_week_model3_posterior.geojson",
)


In [ ]:
# export for prediction
export_idata(idata_nta, "../data/processed/models/model3_nta_idata.nc")